In [1]:
import os
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from google.colab import drive
import sys
import math

## Directory Setup

In [2]:
drive.mount('/content/gdrive/', force_remount=True)

Mounted at /content/gdrive/


In [3]:
def extract_metrics(json_files):
    """
    Extract total distance, total time, success rate, shares, hops, CDV, PDV, and standard deviation of time from multiple JSON files.
    Ensures only valid data is used and avoids incorrect default values.
    """
    total_distances = []
    avg_times = []
    sd_times = []  # Standard deviation of times
    success_rate = []
    shares = []
    hops = []
    cdv_pdv = []

    for file_path in json_files:
        with open(file_path, 'r') as file:
            data = json.load(file)

        if "distances" in data and "times" in data and len(data["times"]) > 0:
            total_distances.append(sum(data["distances"]) / 1000)  # Convert meters to kilometers
            avg_times.append(np.mean(data["times"]))
            sd_times.append(np.std(data["times"]))  # Compute Standard Deviation of time
            success_rate.append(data["success_rate"])
            shares.append(data["shares"])

            # Extracting hops, CDV, and PDV
            if "hops" in data and "CDV" in data and "PDV" in data:
                hops.append(data["hops"])  # Average number of hops per successful delivery
                cdv_pdv.append(data["CDV"] + data["PDV"])  # Sum of CDV and PDV

    # If no valid data is found, return None to indicate the model should be skipped.
    if not total_distances or not avg_times:
        return None

    return {
        "total_distance": np.mean(total_distances),  # Average for line plot
        "average_time": np.mean(avg_times),         # Average for line plot
        "sd_time": np.mean(sd_times),              # Standard Deviation of time
        "all_distances": total_distances,           # List of all values for scatter plot
        "all_avg_times": avg_times,                 # List of all values for scatter plot
        "success_rate":  np.mean(success_rate),
        "shares":  np.mean(shares),
        "hops":  np.mean(hops),                               # List of hops per successful delivery
        "cdv_pdv":  np.mean(cdv_pdv),                          # List of CDV + PDV values
    }



# Method to process city Data
def process_city_data(base_directory, models):
    """
    Process all load and super-hotspot ratio folders in the city's Results directory.
    Ensures only valid models with data are included in results.
    """
    results = {}

    for ratio_folder in os.listdir(base_directory):
        ratio_path = os.path.join(base_directory, ratio_folder, "Repeating")
        if not os.path.isdir(ratio_path):
            continue

        ratio = int(ratio_folder.split('-')[-1])  # Extract ratio from folder name
        results[ratio] = {}

        for model in models:
            # Dynamically select all loads (300, 350, 400, etc.)
            json_files_by_load = {}
            for file_name in os.listdir(ratio_path):
                if file_name.startswith(model) and file_name.endswith(".json"):
                    # Extract load value from the filename (e.g., "-0300-", "-0350-", etc.)
                    load_str = file_name.split('-')[1]  # Extract the second part which contains the load
                    if load_str.isdigit():
                        load_value = int(load_str)
                        json_files_by_load.setdefault(load_value, []).append(os.path.join(ratio_path, file_name))

            # Sort the load values before processing
            sorted_loads = sorted(json_files_by_load.keys())

            # Process each load separately (in sorted order)
            for load_value in sorted_loads:
                json_files = json_files_by_load[load_value]
                metrics = extract_metrics(json_files)  # Keep extract_metrics unchanged

                if metrics:  # Only add models that have valid data
                    if model not in results[ratio]:
                        results[ratio][model] = {}
                    results[ratio][model][load_value] = metrics  # Store by sorted load value

    return results

In [4]:
# from time import thread_time
# Main Execution
cities = ["Philadelphia", "Columbus", "Chicago", "Chicago_mini", "City of New York"]
models = ["results", "baseline1", "baseline2"]
personal_dir = "./gdrive/MyDrive/DeliverAI Data Folder/"
city_results_dict = {}

# Process and Store Results
for city in cities:
    base_directory = os.path.join(personal_dir, f"{city} - RL Delivery Data", "Results")
    city_results = process_city_data(base_directory, models)
    print(f"City: {city}\n")
    # Store city results in the dictionary
    city_results_dict[city] = city_results

City: Philadelphia

City: Columbus

City: Chicago

City: Chicago_mini

City: City of New York



In [7]:
def save_city_results_to_excel(city_results_dict, models, excel_file_path):
    """
    Prepare and save city results, ensuring city name, ratio, load, and model name are in separate columns.
    The load values are now sorted and grouped, and models are always written in the order:
    baseline1 → baseline2 → results
    """
    # Define the row metrics
    metrics = ["success_rate", "total_distance", "average_time", "sd_time", "hops", "cdv_pdv"]

    # Store table data
    table_data = []

    # Header row with separate columns for City, Ratio, Load, Model, and Metrics
    header = ["City", "Ratio", "Load", "Model"] + metrics
    table_data.append(header)

    # Populate the table with values for each city, ratio, model, and sorted load values
    for city in sorted(city_results_dict.keys()):  # Ensure cities are sorted
        for ratio in sorted(city_results_dict[city].keys()):  # Ensure ratios are sorted
            sorted_loads = sorted(set(load for model in models if model in city_results_dict[city][ratio]
                                      for load in city_results_dict[city][ratio][model]))

            for load_value in sorted_loads:  # Iterate in sorted load order
                for model in ["baseline1", "baseline2", "results"]:  # Ensure model order is consistent
                    if model in city_results_dict[city][ratio] and load_value in city_results_dict[city][ratio][model]:
                        metrics_data = city_results_dict[city][ratio][model][load_value]
                        row_data = [city, ratio, load_value, model]  # City, Ratio, Load, Model

                        for metric in metrics:
                            value = metrics_data.get(metric, "N/A")

                            # Formatting values for readability
                            if metric in ["success_rate"]:
                                row_data.append(f"{value:.3f}" if isinstance(value, float) else str(value))
                            elif metric in ["average_time","sd_time", "hops"]:
                                row_data.append(f"{value:.2f}" if isinstance(value, float) else str(value))
                            else:
                                row_data.append(f"{value:.0f}" if isinstance(value, float) else str(value))

                        table_data.append(row_data)

    # Convert to Pandas DataFrame
    df = pd.DataFrame(table_data)

    # Save to Excel
    df.to_excel(excel_file_path, index=False, header=False)

In [8]:

# Call the function to save the Excel file
excel_file_path = "./gdrive/MyDrive/DeliverAI/Graph generation/city_results.xlsx"
save_city_results_to_excel(city_results_dict, models, excel_file_path)